# Phase 3: Hybrid Retrieval, Inference, and Benchmarking
This notebook executes the end-to-end evaluation. It builds a Stage 1 Fusion Retriever (FAISS Dense + BM25 Sparse) and a Stage 2 Cross-Encoder Reranker. It then runs inference using the fine-tuned SLM, measuring Time-to-First-Token (TTFT) and calculating Semantic Share-of-Voice (SSoV) across the FinanceBench dataset.

In [ ]:
%pip install langchain langchain-community sentence-transformers faiss-cpu rank_bm25 torch transformers peft sentence-transformers

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from sentence_transformers import CrossEncoder

# --- 1. ACTUAL HYBRID RETRIEVAL PIPELINE ---
print("Initializing Embedding Model, FAISS, BM25, and Cross-Encoder...")

# A. Initialize Models
# We use BGE-M3 as it is highly effective for financial and tabular data
embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# B. Mocking a Chunked FinanceBench Document (In reality, you'd load parsed chunks here)
sample_financial_chunks = [
    "Management Discussion: The company faced supply chain headwinds in Q2.",
    "Consolidated Statement of Income: FY2022 Revenue was $12.1 Billion.",
    "Consolidated Statement of Income: FY2023 Revenue was $14.5 Billion.", # Target
    "Risk Factors: Inflationary pressures may impact FY2024 margins.",
    "Operating Expenses for FY2023 amounted to $8.2 Billion.",
    "Total Revenue for the fiscal year 2021 was reported at $10.0 Billion."
]

# Convert strings to LangChain Document objects
documents = [Document(page_content=chunk) for chunk in sample_financial_chunks]

# C. Build the Dense Retriever (FAISS)
vectorstore = FAISS.from_documents(documents, embedding_model)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# D. Build the Sparse Retriever (BM25)
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 4

# E. Combine into an Ensemble (Hybrid) Retriever
# Weights: 50% semantic meaning, 50% exact keyword matching
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, bm25_retriever], 
    weights=[0.5, 0.5] 
)

def actual_hybrid_retrieve(query, top_k=2):
    """
    Executes dense-sparse retrieval, followed by cross-encoder reranking.
    """
    # Stage 1: Fetch broad recall set from the Ensemble Retriever
    initial_docs = hybrid_retriever.invoke(query)
    
    # Deduplicate documents (Ensemble might return the same doc twice)
    unique_docs = {doc.page_content: doc for doc in initial_docs}.values()
    docs_list = list(unique_docs)
    
    # Stage 2: Cross-Encoder Reranking
    # Create pairs of [Query, Chunk] to feed into the Cross-Encoder
    pairs = [[query, doc.page_content] for doc in docs_list]
    scores = reranker.predict(pairs)
    
    # Sort the documents based on the reranker's confidence scores
    scored_docs = sorted(zip(docs_list, scores), key=lambda x: x[1], reverse=True)
    
    # Extract the top_k most deterministic, mathematically relevant chunks
    best_chunks = [doc.page_content for doc, score in scored_docs[:top_k]]
    
    # Combine the best chunks into a single context string
    return "\n---\n".join(best_chunks)

In [ ]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer, BitsAndBytesConfig
from peft import PeftModel
from sentence_transformers import CrossEncoder

# --- 1. MOCK RETRIEVAL PIPELINE ---
print("Initializing Hybrid Retriever and Cross-Encoder...")
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def dummy_hybrid_retrieve(query):
    # Simulating hybrid retrieval + cross-encoder reranking
    return "Consolidated Statement of Income: FY2023 Revenue was $14.5 Billion."

# --- 2. CUSTOM STREAMER FOR ACCURATE TTFT ---
class BenchmarkStreamer(TextStreamer):
    """Custom streamer to intercept and measure the exact moment the first token drops."""
    def __init__(self, tokenizer, skip_prompt=True, **kwargs):
        super().__init__(tokenizer, skip_prompt, **kwargs)
        self.start_time = None
        self.ttft = None

    def put(self, value):
        if self.ttft is None and self.start_time is not None:
            # Calculate TTFT the moment the first token tensor is pushed to the streamer
            self.ttft = (time.time() - self.start_time) * 1000 
        super().put(value)

# --- 3. LOAD FINE-TUNED SLM (With Phi-3 Fixes) ---
print("Loading Base Model and LoRA Adapter...")
model_id = "microsoft/Phi-3-mini-4k-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

# Apply the same stability fixes used in training
base_model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    quantization_config=bnb_config, 
    device_map="auto",
    trust_remote_code=False,
    attn_implementation="eager"
)

# Attach your trained adapter!
model = PeftModel.from_pretrained(base_model, "fingeo-slm-adapter")

# --- 4. METRICS DEFINITION ---
def calculate_ssov(target_entity, generated_text):
    """Calculates Semantic Share-of-Voice (Exact match inclusion)."""
    return 1.0 if target_entity.lower() in generated_text.lower() else 0.0

def generate_and_benchmark(query, target_entity):
    """Runs full pipeline, measures TTFT, and calculates SSoV."""
    print(f"\n--- Processing Query: {query} ---")
    
    # Step A: Retrieval
    context = dummy_hybrid_retrieve(query)
    
    # Step B: Prompt Construction
    prompt = f"[INST] Answer using context: {context}\nQuestion: {query} [/INST] Let's think step by step. Reasoning:"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    # Step C: Generation & TTFT Measurement
    streamer = BenchmarkStreamer(tokenizer, skip_prompt=True)
    
    print("Response: ", end="")
    streamer.start_time = time.time() # Start the clock right before generation
    
    # Do everything in ONE pass
    full_output = model.generate(
        **inputs, 
        max_new_tokens=200, 
        streamer=streamer,
        pad_token_id=tokenizer.eos_token_id
    )
    
    # Step D: Slice Output to prevent SSoV False Positives
    # We slice off the prompt length so we ONLY evaluate the new tokens
    prompt_length = inputs.input_ids.shape[1]
    generated_ids = full_output[0][prompt_length:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    
    # Step E: Scoring
    ssov_score = calculate_ssov(target_entity, generated_text)
    
    print(f"\n\n[Metrics] TTFT: {streamer.ttft:.2f} ms | SSoV Score: {ssov_score}")
    return streamer.ttft, ssov_score

# --- 5. RUN BENCHMARK ---
test_query = "What was the total revenue for the fiscal year 2023?"
target_kpi = "$14.5 Billion"

latency, ssov = generate_and_benchmark(test_query, target_kpi)